In [2]:
%pip install -qU langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [5]:
AZURE_OPENAI_API_KEY = 'c739981ee79541deb8414f0bd8bb576c'
AZURE_OPENAI_ENDPOINT = 'https://tcl-azure-westus3.openai.azure.com'

### instantiation

In [7]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    api_key = AZURE_OPENAI_API_KEY,
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    azure_deployment = "tcl-gpt4o1",
    api_version = "2024-04-01-preview",
    temperature = 0
)

### invocation

In [8]:
message = [
    (
        "system",
        "You are a helpful assistant that translates English to Chinese. Translate the user sentence.",
    ),
    ("human","I love programming"),
]
ai_msg = llm.invoke(message)
ai_msg

AIMessage(content='我喜欢编程', response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 30, 'total_tokens': 34}, 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_abc28019ad', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}, id='run-cb07b8f7-94d9-4770-98a7-e53320eb930c-0', usage_metadata={'input_tokens': 30, 'output_tokens': 4, 'total_tokens': 34})

In [9]:
print(f'AI:{ai_msg.content}')

AI:我喜欢编程


### Chaining

In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant that translates {input_language} to {output_language}."
    ),
    ("human","{input}"),
])

chain = prompt | llm
chain.invoke({
    "input_language":"English",
    "output_language":"Chinese",
    "input":"I love programming!"
})

AIMessage(content='我喜欢编程！', response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 26, 'total_tokens': 31}, 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_abc28019ad', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}, id='run-e9a69762-57b6-4cfe-9a6c-9293620764f1-0', usage_metadata={'input_tokens': 26, 'output_tokens': 5, 'total_tokens': 31})

### Specifying model version(指定模型版本)

In [13]:
%pip install -qU langchain-community

Note: you may need to restart the kernel to use updated packages.


In [14]:
from langchain_community.callbacks import get_openai_callback

with get_openai_callback() as cb:
    llm.invoke(message)
    print(
        f"Total Cost (USD): ${format(cb.total_cost,'.6f')}"
    )

Total Cost (USD): $0.000210


In [19]:
llm_0301 = AzureChatOpenAI(
    api_key = AZURE_OPENAI_API_KEY,
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    azure_deployment = "tcl-gpt4o1",
    api_version = "2024-04-01-preview",
    temperature = 0,
    model_version = "0301",
)
with get_openai_callback() as cb:
    llm_0301.invoke(message)
    print(
        f"Total Cost (USD): ${format(cb.total_cost,'.6f')}"
    )    

Total Cost (USD): $0.000000


### AzureMLChatOnlineEndpoint(自定义模型聊天)

In [ ]:
from langchain_community.chat_models.azureml_endpoint import (
    AzureMLEndpointApiType,
    CustomOpenAIChatContentFormatter,
)
from langchain_core.messages import HumanMessage

chat = AzureMLChatOnlineEndpoint(
    endpoint_url="https://<your-endpoint>.<your_region>.inference.ml.azure.com/score",
    endpoint_api_type=AzureMLEndpointApiType.dedicated,
    endpoint_api_key="my-api-key",
    content_formatter=CustomOpenAIChatContentFormatter(),
)
response = chat.invoke(
    [HumanMessage(content="Will the Collatz conjecture ever be solved?")]
)
response

In [ ]:
chat = AzureMLChatOnlineEndpoint(
    endpoint_url="https://<your-endpoint>.<your_region>.inference.ml.azure.com/v1/chat/completions",
    endpoint_api_type=AzureMLEndpointApiType.serverless,
    endpoint_api_key="my-api-key",
    content_formatter=CustomOpenAIChatContentFormatter,
)
response = chat.invoke(
    [HumanMessage(content="Will the Collatz conjecture ever be solved?")]
)
response

In [ ]:
response = chat.invoke(
    [HumanMessage(content="Will the Collatz conjecture ever be solved?")],
    max_tokens=512,
)
response